# 3 — Balanced Stage 2 smoke test
Runs six episodes: horizons 20/25/30, Native/+200 ms, spatial task, seed 5. They are real manifest episodes and will be retained by the resumable full run.


In [ ]:
import os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; PY=Path.home()/"venv-stage1-id/bin/python"; OUT=Path.home()/"stage2"; GPU=(Path.home()/"stage2_gpu.txt").read_text().strip()
env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1"})
cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_stage2","--config",str(R/"async_vla_benchmark/configs/stage2.yaml"),"--manifest",str(OUT/"stage2_local_sensitivity_manifest.csv"),"--output-dir",str(OUT),"--task","spatial_transport","--seed","5","--horizon","20","--horizon","25","--horizon","30","--delay","0","--delay","200","--resume","--verbose"]
log=OUT/"stage2_smoke.log"
with open(log,"ab") as fh: subprocess.run(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,check=True)
print(''.join(log.read_text().splitlines(True)[-30:]))


In [ ]:
import csv
rows=list(csv.DictReader(open(OUT/"stage2_local_sensitivity_manifest.csv"))); smoke={r['run_id'] for r in rows if r['task_key']=='spatial_transport' and r['seed']=='5' and r['configured_n_action_steps'] in {'20','25','30'} and r['added_delay_ms'] in {'0','200'}}
done={p.stem for p in (OUT/"episodes").glob("*.json")}; print("smoke artifacts",len(smoke & done),"/ 6")
if len(smoke & done)!=6: raise SystemExit("STOP: smoke is incomplete")
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.validate_stage2","--manifest",str(OUT/"stage2_local_sensitivity_manifest.csv"),"--output-dir",str(OUT),"--allow-incomplete"],cwd=R,check=True)
print("STOP HERE and review the log tail and validator output before notebook 04.")
